# CP-HK transfer-in: NM source models -> CP-HK

This notebook processes the matched-checkpoint CP-HK transfer-in run. It compares NM-trained source models from the recent UKR/COVID, COVID/Midterm, and TwiBot-20 experiments when evaluated on the new **CP-HK** retweet graph.

Target task: neighbor matching, 3-shot, evaluated as both 3-way and 30-way. Merged models use the matched 50k checkpoint, not the full 110k checkpoint.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("cp_hk_transfer_in_metrics.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA)

LABEL = {
    "nm_matrix_ukr": "ukr\n(single)",
    "nm_matrix_covid": "covid\n(single)",
    "nm_matrix_merged_match": "ukr+cov\n(prop)",
    "nm_xsrc_within_source_match": "ukr+cov\n(within)",
    "nm_cm_covid": "covid\n(single)",
    "nm_cm_midterm": "midterm\n(single)",
    "nm_cm_merged_match": "cov+mid\n(prop)",
    "nm_cm_within_match": "cov+mid\n(within)",
    "nm_cm_within_balanced_match": "cov+mid\n(within-bal)",
    "nm_twibot20": "twibot\n(single)",
}

STRAT_COLOR = {
    "single": "#8a8f98",
    "merged_proportional_matched": "#2f6f9f",
    "merged_within_matched": "#5a9f72",
    "merged_within_balanced_matched": "#b45f4d",
}

ORDER = {
    "ukr_covid": [
        "nm_matrix_ukr",
        "nm_matrix_covid",
        "nm_matrix_merged_match",
        "nm_xsrc_within_source_match",
    ],
    "covid_midterm": [
        "nm_cm_covid",
        "nm_cm_midterm",
        "nm_cm_merged_match",
        "nm_cm_within_match",
        "nm_cm_within_balanced_match",
    ],
    "twibot20": ["nm_twibot20"],
}

df["label"] = df["model"].map(LABEL).fillna(df["model"])
df.head()

In [ ]:
summary = df.pivot_table(
    index=["experiment", "source", "strategy", "model"],
    columns="n_way",
    values=["accuracy", "f1", "roc_auc"],
)
display(summary.round(4))

print("Top 3-way accuracy")
display(df[df.n_way.eq(3)].sort_values("accuracy", ascending=False)[["model", "source", "strategy", "accuracy", "roc_auc"]].round(4).reset_index(drop=True))

print("Top 30-way accuracy")
display(df[df.n_way.eq(30)].sort_values("accuracy", ascending=False)[["model", "source", "strategy", "accuracy", "roc_auc"]].round(4).reset_index(drop=True))

Chance accuracy is **0.333** for 3-way NM and **0.033** for 30-way NM. ROC-AUC chance is **0.5**. Bars are colored by strategy: grey = single source, blue = merged proportional, green = merged within-source, red = merged within-balanced.

In [ ]:
def style_axis(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#d7d7d7", linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)

def annotate(ax, bars, fmt="{:.3f}"):
    ymin, ymax = ax.get_ylim()
    pad = (ymax - ymin) * 0.02
    for b in bars:
        v = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, v + pad, fmt.format(v), ha="center", va="bottom", fontsize=8)

def panel(ax, exp, n_way, metric, title, chance=None, ylim=None):
    order = ORDER[exp]
    sub = df[df.experiment.eq(exp) & df.n_way.eq(n_way)].set_index("model").reindex(order)
    bars = ax.bar(
        range(len(order)),
        sub[metric].values,
        color=[STRAT_COLOR[s] for s in sub["strategy"]],
        width=0.72,
    )
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([LABEL[m] for m in order], fontsize=8)
    if ylim:
        ax.set_ylim(*ylim)
    if chance is not None:
        ax.axhline(chance, color="#333333", ls="--", lw=1.1)
        ax.text(0.02, chance, f" chance={chance:.3f}", transform=ax.get_yaxis_transform(), va="bottom", fontsize=8)
    ax.set_title(title, fontsize=10)
    style_axis(ax)
    annotate(ax, bars)

def plot_grid(metric, ylabel, filename, ylims):
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.0))
    for row, n_way in enumerate([3, 30]):
        chance = 1 / n_way if metric == "accuracy" else 0.5
        panel(axes[row, 0], "ukr_covid", n_way, metric, f"UKR / COVID sources, {n_way}-way", chance=chance, ylim=ylims[n_way])
        panel(axes[row, 1], "covid_midterm", n_way, metric, f"COVID / Midterm sources, {n_way}-way", chance=chance, ylim=ylims[n_way])
        axes[row, 0].set_ylabel(ylabel)
    fig.suptitle(f"CP-HK transfer-in: {ylabel}", y=1.01)
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    return fig


In [ ]:
plot_grid(
    "accuracy",
    "accuracy",
    "cp_hk_transfer_in_accuracy.png",
    {3: (0.32, 0.56), 30: (0.03, 0.18)},
);

In [ ]:
plot_grid(
    "roc_auc",
    "ROC-AUC",
    "cp_hk_transfer_in_roc_auc.png",
    {3: (0.64, 0.78), 30: (0.62, 0.74)},
);

In [ ]:
wide = df.pivot(index=["model", "source", "strategy", "experiment"], columns="n_way", values=["accuracy", "roc_auc"])
wide.columns = [f"{metric}_{n_way}way" for metric, n_way in wide.columns]
wide = wide.reset_index()
wide["accuracy_drop_3_to_30"] = wide["accuracy_3way"] - wide["accuracy_30way"]
wide["auc_drop_3_to_30"] = wide["roc_auc_3way"] - wide["roc_auc_30way"]
display(wide.sort_values("roc_auc_30way", ascending=False).round(4))

fig, ax = plt.subplots(figsize=(6.4, 5.0))
for _, row in wide.iterrows():
    ax.scatter(row["roc_auc_3way"], row["roc_auc_30way"], s=70, color=STRAT_COLOR[row["strategy"]], edgecolor="white", linewidth=0.8)
    ax.text(row["roc_auc_3way"] + 0.001, row["roc_auc_30way"] + 0.001, row["source"], fontsize=8)
ax.set_xlabel("3-way ROC-AUC")
ax.set_ylabel("30-way ROC-AUC")
ax.set_title("CP-HK transfer-in: easy vs hard NM eval")
style_axis(ax)
fig.tight_layout()
fig.savefig(FIG_DIR / "cp_hk_transfer_in_auc_scatter.png", dpi=220, bbox_inches="tight")